# Этап 5. Финальный пайплайн V3

Вопрос, гибридный поиск (BM25 + E5 + CLIP, RRF, BGE), ответ Qwen2.5 со ссылками на пункты ПУЭ.

Нужны: Milvus с коллекциями V2 `diplom_v2_e5` и `diplom_v2_clip`, Ollama с моделью `qwen2.5:14b-instruct-q4_K_M`, настройки в `configs/v3_config.json` (device cuda).

In [ ]:
import sys, json
from pathlib import Path
V3 = Path.cwd()
while not (V3 / "configs/v3_config.json").is_file():  # ноутбук лежит в V3/docs/notebooks
    if V3 == V3.parent:
        raise FileNotFoundError("Не найдена папка V3 (configs/v3_config.json)")
    V3 = V3.parent
for p in (V3 / "src", V3 / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"  # графики видны и после сохранения

In [ ]:
from retrieval_v3.hybrid_retriever import HybridRetrieverV3
from retrieval_v3.rag_pipeline import V3RAGPipeline

retriever = HybridRetrieverV3(V3 / "configs/v3_config.json")
rag = V3RAGPipeline()
print("Чанков:", len(retriever.chunks), "| стеммер BM25:", retriever.tokenizer.kind)

## Вопрос

Шаг поиска задаётся строкой `VARIANT`: `bm25`, `e5`, `bm25_e5`, `bm25_e5_clip` и добавки `+rerank`, `+fullchunk`, `+exact`.

In [ ]:
QUESTION = "Какие виды испытаний силовых трансформаторов перечислены в п. 1.8.16?"
VARIANT = "bm25_e5_clip+rerank+exact"
TOP_K = 5

result = rag.ask(QUESTION, retriever, variant=VARIANT, top_k=TOP_K)
import pandas as pd
display(pd.DataFrame(result["retrieved"]))
print(result["answer"])

## Только поиск, без LLM

In [ ]:
for r in retriever.search(QUESTION, variant=VARIANT, limit=TOP_K):
    print(r["rank"], r["metadata"].get("Clause"), "| по номеру" if r["exact_clause"] else "", "|", r["content"][:200].replace("\n", " "))